In [ ]:
from pathlib import Path
from pd_estim_A.data.data_import import (
    load_data, load_ecb_1y_yield,
    fill_liabilities, drop_high_leverage_firms,
    prepare_merton_inputs, prepare_nig_inputs
)

# Paths
print(Path.cwd())
data_path = Path.cwd() / ".." / "data" / "raw"
output_path = Path.cwd() / ".." / "data" / "derived"

# Load data and prepare panels
ret_daily, bs, coverage = load_data(
    data_path / "Jan2025_Accenture_Dataset_ErasmusCase.xlsx",
    start_date="2012-01-01",
    end_date="2025-12-19",
    enforce_coverage=True,
    coverage_tol=0.995,
    liabilities_scale="auto",
    verbose=True,
)

df_rf = load_ecb_1y_yield(
    startPeriod="2010-01-01",
    endPeriod="2025-12-31",
    out_file= output_path / "ecb_yc_1y_aaa.xml",
    verify_ssl=True,  # recommended if it works
)

df_cal = ret_daily[["date"]].drop_duplicates().sort_values("date").reset_index(drop=True)

debt_daily = fill_liabilities(bs, df_cal)

ret_filt, bs_filt, lev_by_firm, dropped = drop_high_leverage_firms(
    ret_daily,
    bs,
    df_calendar=df_cal,
    debt_daily=debt_daily,
    lev_threshold=8.0,
    lev_agg="median",
    verbose=True,
)

# keep debt panel consistent with filtered firms
keep = set(ret_filt["gvkey"].astype(str).unique())
debt_daily_filt = debt_daily[debt_daily["gvkey"].astype(str).isin(keep)].copy()

# Merton
merton_df = prepare_merton_inputs(ret_filt, bs_filt, df_rf, debt_daily=debt_daily_filt)

# NIG
nig_df, em_cache = prepare_nig_inputs(ret_filt, bs_filt, df_rf, debt_daily=debt_daily_filt, build_em=False)

In [6]:
print(merton_df.describe())
print("******************************")
print(nig_df.describe())


                                date             E    logret_mcap  \
count                         131155  1.311550e+05  131155.000000   
mean   2018-12-26 16:17:25.016964864  7.470086e+10       0.000339   
min              2012-01-03 00:00:00  1.215293e+09      -0.265878   
25%              2015-06-30 00:00:00  3.097716e+10      -0.007498   
50%              2018-12-27 00:00:00  5.218637e+10       0.000189   
75%              2022-06-23 12:00:00  8.899452e+10       0.008511   
max              2025-12-19 00:00:00  5.901056e+11       0.501710   
std                              NaN  7.517578e+10       0.016869   

                   r             B  sigma_E_daily        sigma_E  
count  131155.000000  1.311550e+05  126655.000000  126655.000000  
mean        0.002988  7.068479e+10       0.016030       0.254476  
min        -0.009130  7.555000e+08       0.007427       0.117892  
25%        -0.006733  1.412770e+10       0.012293       0.195147  
50%        -0.002542  3.579000e+10       0.